In [ ]:
! pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [ ]:
validation_data = pd.read_csv('samsum-validation.csv')
train_data = pd.read_csv('samsum-train.csv')

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
# random sampling

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

## data pre-processing

In [ ]:
import re

def clean_data(text):
    text = re.sub(r"\r\n|\r|\n", " ", text)   # replace all line breaks with a space
    text = re.sub(r"\s+", " ", text)          # collapse multiple spaces
    text = re.sub(r"<.*?>", "", text)         # remove HTML tags
    return text.strip().lower()



In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:  claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

## Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs


In [ ]:
train_dataset = train_data.to_dict('records')
val_dataset = validation_data.to_dict('records')

### Custom Dataset for Tokenized Data

To ensure the `Trainer` receives tokenized inputs, we need a custom PyTorch `Dataset` class. This class will handle the tokenization of the dialogue and summary text on demand, producing `input_ids`, `attention_mask`, and `labels` (which are the tokenized summary `input_ids`).

In [ ]:
import torch
from torch.utils.data import Dataset

class SummarizationDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length_input=512, max_length_target=150):
        self.tokenizer = tokenizer
        self.data_list = data_list
        self.max_length_input = max_length_input
        self.max_length_target = max_length_target

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        dialogue = item['dialogue']
        summary = item['summary']

        # Tokenize the dialogue (input) - return lists of IDs
        inputs = self.tokenizer(dialogue, padding="max_length", max_length=self.max_length_input, truncation=True)
        # Tokenize the summary (target) - return lists of IDs
        targets = self.tokenizer(summary, padding="max_length", max_length=self.max_length_target, truncation=True)

        # Add labels. For sequence-to-sequence models, labels are the tokenized target input_ids.
        # Return as lists of integers, DataCollator will convert to tensors and pad.
        return {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask'],
            'labels': targets['input_ids']
        }


### Instantiate Tokenized Datasets

Now, let's create instances of our `SummarizationDataset` for the training and validation data, passing in the `tokenizer`.

In [ ]:
# Instantiate the tokenized datasets
train_dataset_tokenized = SummarizationDataset(train_dataset, tokenizer)
val_dataset_tokenized = SummarizationDataset(val_dataset, tokenizer)

print(f"Number of training samples: {len(train_dataset_tokenized)}")
print(f"Number of validation samples: {len(val_dataset_tokenized)}")

# Display a sample from the tokenized dataset to confirm structure
print("\nSample from tokenized training dataset:")
sample = train_dataset_tokenized[0]
for k, v in sample.items():
    print(f"{k}: {v.shape}")

Number of training samples: 4000
Number of validation samples: 500

Sample from tokenized training dataset:
input_ids: torch.Size([512])
attention_mask: torch.Size([512])
labels: torch.Size([150])


### Update Trainer Initialization

Finally, we need to modify the `Trainer` to use these newly created tokenized datasets.

## Working with our model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
train_dataset[0]

{'id': '13811908',
 'dialogue': "violet: hi! i came across this austin's article and i thought that you might find it interesting violet:  claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)",
 'summary': "violet sent claire austin's article."}

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)


In [ ]:
from transformers import DataCollatorForSeq2Seq

# Initialize data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized, # Use the tokenized dataset
    eval_dataset=val_dataset_tokenized,     # Use the tokenized dataset
    data_collator=data_collator # Explicitly pass the data collator
)

In [ ]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
# model load => fine-tune => save the model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './saved_summary_model'.

## Test the core logic for summarization

In [ ]:

def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)